# Track 2 — Fine-tune Qwen2.5-7B NER (Viettel AI Race)
Runtime > Change runtime type > **T4 GPU**. Rồi Run all.
~30-60 phút. Cuối cùng tải `qwen_ner_lora.zip` về.

In [ ]:
# 1. Clone repo + cài package
!git clone -q https://github.com/quocbao271207/Viettel.git
%cd Viettel
!cp dev/track2/train.jsonl dev/track2/dev.jsonl .
!pip install -q -U 'transformers>=4.44' peft trl datasets accelerate bitsandbytes
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU — đổi runtime!')

In [ ]:
# 2. Train QLoRA Qwen2.5-7B-Instruct (generative NER)
import json, torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
MODEL='Qwen/Qwen2.5-7B-Instruct'
tok=AutoTokenizer.from_pretrained(MODEL)
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
model=AutoModelForCausalLM.from_pretrained(MODEL,quantization_config=bnb,device_map='auto')
model=get_peft_model(model,LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))
model.print_trainable_parameters()
ds=load_dataset('json',data_files={'train':'train.jsonl','dev':'dev.jsonl'})
ds=ds.map(lambda ex:{'text':tok.apply_chat_template(ex['messages'],tokenize=False,add_generation_prompt=False)})
tr=SFTTrainer(model=model,tokenizer=tok,train_dataset=ds['train'],eval_dataset=ds['dev'],
  args=SFTConfig(output_dir='qwen_ner_lora',max_seq_length=4096,per_device_train_batch_size=1,
    gradient_accumulation_steps=8,num_train_epochs=6,learning_rate=2e-4,warmup_ratio=0.05,
    logging_steps=5,eval_strategy='epoch',save_strategy='epoch',bf16=True,gradient_checkpointing=True,report_to='none'))
tr.train(); tr.save_model('qwen_ner_lora'); print('TRAIN XONG')

In [ ]:
# 3. Test nhanh 1 file + đóng gói adapter để tải về
import json
raw=open('input/1.txt',encoding='utf-8').read()
SYS='Bạn là chuyên gia gán nhãn NER y khoa tiếng Việt. Trích MỌI khái niệm y tế (THUỐC/CHẨN_ĐOÁN/TRIỆU_CHỿNG), mỗi lần nhắc một mục, kèm assertion. Chỉ in JSON array [{"text","type","assertion"}].'.replace('Ỿ','Ứ')
ids=tok.apply_chat_template([{'role':'system','content':SYS},{'role':'user','content':raw}],add_generation_prompt=True,return_tensors='pt').to(model.device)
out=model.generate(ids,max_new_tokens=2048,do_sample=False)
print(tok.decode(out[0][ids.shape[1]:],skip_special_tokens=True)[:1500])
!zip -qr qwen_ner_lora.zip qwen_ner_lora
from google.colab import files; files.download('qwen_ner_lora.zip')